In [ ]:
# ============================================================================
# LUMIS-1 RUNPOD ENVIRONMENT SETUP
# Target: RunPod PyTorch 2.4.0 Template (NVIDIA A40/A100, CUDA 12.4)
# ============================================================================

import subprocess
import sys

def run(cmd, desc):
    """Run shell command with progress output."""
    print(f"\n{'='*60}")
    print(f"[LUMIS-1] {desc}")
    print(f"{'='*60}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print(f"STDERR: {result.stderr}")
    return result.returncode == 0

# ============================================================================
# PHASE 1: GPU VERIFICATION
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 1: GPU VERIFICATION")
print("#"*60)

run("nvidia-smi", "Checking NVIDIA GPU...")
run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv", "GPU Details")

# ============================================================================
# PHASE 2: DEPENDENCY INSTALLATION
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 2: DEPENDENCY INSTALLATION (using pre-installed PyTorch)")
print("#"*60)

# Upgrade pip first
run(f"{sys.executable} -m pip install --upgrade pip -q", "Upgrading pip...")

# Install HuggingFace ecosystem
run(
    f"{sys.executable} -m pip install "
    f"transformers>=4.40.0 "
    f"datasets>=2.18.0 "
    f"accelerate>=0.28.0 "
    f"peft>=0.10.0 "
    f"-q",
    "Installing transformers, datasets, accelerate, peft..."
)

# Install ONNX runtime and Optimum
run(
    f"{sys.executable} -m pip install "
    f"onnxruntime-gpu>=1.17.0 "
    f"optimum>=1.17.0 "
    f"-q",
    "Installing onnxruntime-gpu, optimum..."
)

# Print installed versions
print("\n" + "="*60)
print("[LUMIS-1] Installed Package Versions")
print("="*60)

import torch
import transformers
import datasets
import accelerate
import peft
import onnxruntime
from importlib.metadata import version as pkg_version

print(f"torch:           {torch.__version__}")
print(f"transformers:    {transformers.__version__}")
print(f"datasets:        {datasets.__version__}")
print(f"accelerate:      {accelerate.__version__}")
print(f"peft:            {peft.__version__}")
print(f"onnxruntime:     {onnxruntime.__version__}")
print(f"optimum:         {pkg_version('optimum')}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version:    {torch.version.cuda}")
    print(f"GPU device:      {torch.cuda.get_device_name(0)}")

# ============================================================================
# PHASE 3: MODEL LOADING & VERIFICATION
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 3: MODEL LOADING & VERIFICATION")
print("#"*60)

from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

models = {}
tokenizers = {}

# --- Speaker Model: IBM Granite 4.0-H-1B ---
print("\n" + "="*60)
print("[LUMIS-1] Loading Speaker: ibm-granite/granite-4.0-h-1b")
print("="*60)

speaker_id = "ibm-granite/granite-4.0-h-1b"
tokenizers["speaker"] = AutoTokenizer.from_pretrained(speaker_id, trust_remote_code=True)
models["speaker"] = AutoModelForCausalLM.from_pretrained(
    speaker_id,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True,
)
models["speaker"].eval()
print(f"Speaker loaded on: {models['speaker'].device}")
print(f"Speaker dtype: {models['speaker'].dtype}")

# --- Safety Model: unitary/toxic-bert ---
print("\n" + "="*60)
print("[LUMIS-1] Loading Safety: unitary/toxic-bert")
print("="*60)

safety_id = "unitary/toxic-bert"
tokenizers["safety"] = AutoTokenizer.from_pretrained(safety_id)
models["safety"] = AutoModelForSequenceClassification.from_pretrained(safety_id).to("cuda")
models["safety"].eval()
print(f"Safety loaded on: {next(models['safety'].parameters()).device}")
print(f"Safety labels: {models['safety'].config.id2label}")

# --- NLI Model: cross-encoder/nli-deberta-v3-xsmall ---
print("\n" + "="*60)
print("[LUMIS-1] Loading NLI: cross-encoder/nli-deberta-v3-xsmall")
print("="*60)

nli_id = "cross-encoder/nli-deberta-v3-xsmall"
tokenizers["nli"] = AutoTokenizer.from_pretrained(nli_id)
models["nli"] = AutoModelForSequenceClassification.from_pretrained(nli_id).to("cuda")
models["nli"].eval()
print(f"NLI loaded on: {next(models['nli'].parameters()).device}")
print(f"NLI labels: {models['nli'].config.id2label}")

# ============================================================================
# PHASE 4: INFERENCE TESTS
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 4: INFERENCE TESTS")
print("#"*60)

# --- Test Speaker Generation ---
print("\n" + "="*60)
print("[LUMIS-1] Test: Speaker Generation")
print("="*60)

test_prompt = "What is the capital of France?"
messages = [{"role": "user", "content": test_prompt}]

input_ids = tokenizers["speaker"].apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output_ids = models["speaker"].generate(
        input_ids,
        max_new_tokens=64,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizers["speaker"].eos_token_id,
    )

response = tokenizers["speaker"].decode(output_ids[0], skip_special_tokens=True)
print(f"Prompt: {test_prompt}")
print(f"Response: {response}")

# --- Test Safety Classification ---
print("\n" + "="*60)
print("[LUMIS-1] Test: Safety Classification")
print("="*60)

test_texts = [
    "Hello, how are you today?",
    "I hate everything about this.",
]

for text in test_texts:
    inputs = tokenizers["safety"](text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        logits = models["safety"](**inputs).logits
        probs = torch.sigmoid(logits)[0]
    
    print(f"\nText: \"{text}\"")
    for i, label in models["safety"].config.id2label.items():
        print(f"  {label}: {probs[i].item():.4f}")

# --- Test NLI Classification ---
print("\n" + "="*60)
print("[LUMIS-1] Test: NLI Classification")
print("="*60)

nli_pairs = [
    ("The capital of France is Paris.", "Paris is a city in France."),
    ("The sky is blue.", "The sky is green."),
]

for premise, hypothesis in nli_pairs:
    inputs = tokenizers["nli"](premise, hypothesis, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    with torch.no_grad():
        logits = models["nli"](**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]
    
    print(f"\nPremise: \"{premise}\"")
    print(f"Hypothesis: \"{hypothesis}\"")
    for i, label in models["nli"].config.id2label.items():
        print(f"  {label}: {probs[i].item():.4f}")

# ============================================================================
# PHASE 5: VRAM USAGE REPORT
# ============================================================================
print("\n" + "#"*60)
print("# PHASE 5: VRAM USAGE REPORT")
print("#"*60)

allocated = torch.cuda.memory_allocated() / 1024**3
reserved = torch.cuda.memory_reserved() / 1024**3
max_allocated = torch.cuda.max_memory_allocated() / 1024**3

print(f"\nVRAM Allocated:     {allocated:.2f} GB")
print(f"VRAM Reserved:      {reserved:.2f} GB")
print(f"VRAM Max Allocated: {max_allocated:.2f} GB")

# Final nvidia-smi for verification
run("nvidia-smi", "Final GPU State")

# ============================================================================
# SETUP COMPLETE
# ============================================================================
print("\n" + "#"*60)
print("# LUMIS-1 ENVIRONMENT SETUP COMPLETE")
print("#"*60)
print("\nAll models loaded and verified on GPU.")
print("Ready for Lumis-1 development.")